# FEDformer — Optuna Hyperparameter Search (NVDA)

Ejecuta dos estudios Optuna (composite vs sharpe) secuencialmente.
Los resultados se guardan en `/kaggle/working/` como SQLite + log.

**Requisitos Kaggle:**
- Settings -> Accelerator: **GPU T4 × 2**
- Settings -> Internet: **ON**

In [ ]:
# ── Celda 1: Verificar GPU ────────────────────────────────────────────────
import torch

n_gpus = torch.cuda.device_count()
print(f"GPUs disponibles: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  SMs: {props.multi_processor_count}  VRAM: {props.total_memory // 1024**3} GB")

if n_gpus == 0:
    print("⚠ No hay GPU disponible — los runs serán lentos en CPU")
    print("  Activa GPU T4 en Settings -> Accelerator")

In [ ]:
# ── Celda 2: Instalar dependencias no incluidas en Kaggle ─────────────────
# torch, pandas, numpy, sklearn, scipy, matplotlib ya están preinstalados
!pip install -q optuna pandas-ta-classic yfinance

In [ ]:
# ── Celda 3: Clonar repositorio ───────────────────────────────────────────
import os
import sys

REPO = (
    "https://github.com/RubenPanero/FEDformer-Probabilistic-Time-Series-Forecasting.git"
)
WORKDIR = "/kaggle/working/fedformer"

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
else:
    print("Repo ya clonado — actualizando")
    !git -C {WORKDIR} pull

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print(f"Directorio de trabajo: {os.getcwd()}")

In [ ]:
# ── Celda 4: Verificar estructura y datos ─────────────────────────────────
import os
import pandas as pd

dataset_path = "data/NVDA_features.csv"
if not os.path.exists(dataset_path):
    print("Descargando dataset (yfinance mock) porque no está en git...")
    !python -m data.financial_dataset_builder --symbol NVDA --output_dir data --use_mock

df = pd.read_csv("data/NVDA_features.csv")
print(f"NVDA dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Rango fechas: {df.iloc[0, 0]} -> {df.iloc[-1, 0]}")

In [ ]:
# ── Celda 5: Estudio 1 — Composite (≈ 40min en T4) ───────────────────────
import subprocess
import sys
import time

STORAGE_COMPOSITE = "/kaggle/working/nvda_composite_v1.db"

print("Iniciando estudio composite...")
t0 = time.time()

result = subprocess.run(
    [
        sys.executable,
        "tune_hyperparams.py",
        "--csv",
        "data/NVDA_features.csv",
        "--n-trials",
        "10",
        "--study-objective",
        "composite",
        "--storage-path",
        STORAGE_COMPOSITE,
    ],
    env={**os.environ, "MPLBACKEND": "Agg"},
)

elapsed = (time.time() - t0) / 60
print(f"Composite finalizado en {elapsed:.1f} min — exit code: {result.returncode}")

In [ ]:
# ── Celda 6: Estudio 2 — Sharpe puro (≈ 40min en T4) ────────────────────
import subprocess
import sys
import time

STORAGE_SHARPE = "/kaggle/working/nvda_sharpe_v1.db"

print("Iniciando estudio sharpe...")
t0 = time.time()

result = subprocess.run(
    [
        sys.executable,
        "tune_hyperparams.py",
        "--csv",
        "data/NVDA_features.csv",
        "--n-trials",
        "10",
        "--study-objective",
        "sharpe",
        "--storage-path",
        STORAGE_SHARPE,
    ],
    env={**os.environ, "MPLBACKEND": "Agg"},
)

elapsed = (time.time() - t0) / 60
print(f"Sharpe finalizado en {elapsed:.1f} min — exit code: {result.returncode}")

In [ ]:
# ── Celda 7: Comparación de resultados ───────────────────────────────────
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

composite_study = optuna.load_study(
    study_name="tune_NVDA_features",
    storage=f"sqlite:///{STORAGE_COMPOSITE}",
)
sharpe_study = optuna.load_study(
    study_name="tune_NVDA_features",
    storage=f"sqlite:///{STORAGE_SHARPE}",
)


def summarize(study, name):
    best = study.best_trial
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    print(f"\n{'=' * 55}")
    print(f"  {name}")
    print(f"{'=' * 55}")
    print(f"  Trials completados : {len(completed)} | podados: {len(pruned)}")
    print(f"  Mejor score        : {best.value:.4f}")
    print("  Parámetros óptimos :")
    for k, v in best.params.items():
        print(f"    {k:<25} {v}")
    ua = best.user_attrs
    print("  ── Métricas del mejor trial ──")
    for key in [
        "sharpe",
        "sortino",
        "var_95",
        "pinball_p50",
        "coverage_80",
        "composite_score",
    ]:
        val = ua.get(key, float("nan"))
        print(f"    {key:<25} {val:.4f}")


summarize(composite_study, "COMPOSITE  (0.5·Sharpe + 0.3·pinball + 0.2·coverage)")
summarize(sharpe_study, "SHARPE PURO")

# Criterios de éxito
bc = composite_study.best_trial.user_attrs
bs = sharpe_study.best_trial.user_attrs
print("\n" + "=" * 55)
print("  CRITERIOS DE ÉXITO")
print("=" * 55)
sharpe_ok = bc.get("sharpe", 0) >= bs.get("sharpe", 0) * 0.95
pinball_ok = bc.get("pinball_p50", 1) <= bs.get("pinball_p50", 1)
coverage_ok = bc.get("coverage_80", 0) >= 0.75
print(f"  Composite no sacrifica >5% Sharpe : {'✓' if sharpe_ok else '✗'}")
print(f"  Composite mejora pinball_p50       : {'✓' if pinball_ok else '✗'}")
print(f"  coverage_80 composite ≥ 0.75       : {'✓' if coverage_ok else '✗'}")

In [ ]:
# ── Celda 8: Guardar resumen en JSON (para descargar) ────────────────────
import json


def trial_to_dict(trial):
    return {
        "best_score": trial.value,
        "params": trial.params,
        "metrics": trial.user_attrs,
    }


summary = {
    "composite": trial_to_dict(composite_study.best_trial),
    "sharpe": trial_to_dict(sharpe_study.best_trial),
}

out_path = "/kaggle/working/optuna_comparison.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Resumen guardado en {out_path}")
print(json.dumps(summary, indent=2))